In [163]:
from dataclasses import dataclass

import pandas as pd
import numpy as np

In [164]:
data = pd.read_csv('marketing_campaign_dataset.csv')
print(f'dataset loaded with {data.shape[0]} rows and {data.shape[1]} columns')

dataset loaded with 2020 rows and 12 columns


In [165]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   str    
 1   Campaign_Name  2020 non-null   str    
 2   Start_Date     2020 non-null   str    
 3   End_Date       2020 non-null   str    
 4   Channel        1919 non-null   str    
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   str    
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   str    
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 189.5 KB


In [166]:
data.head(10)

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
5,CMP-00006,Q4_BlackFriday_CMP-00006,2023-10-15 00:00:00,2023-10-28,Instagram,83386,2643,2697.03,NaN,1,NaN,IN
6,CMP-00007,Q3_Launch_CMP-00007,2023-10-07 00:00:00,2023-10-23,Facebook,38194,1135,1232.76,178.0,Yes,NaN,FA
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,Instagram,88498,1173,865.7,127.0,1,NaN,IN
8,CMP-00009,Q4_BlackFriday_CMP-00009,2023-03-23 00:00:00,2023-04-01,Google Ads,45131,1179,1046.18,104.0,1,NaN,GO
9,CMP-00010,Q2_Winter_CMP-00010,2023-03-21 00:00:00,2023-04-01,Email,61263,1153,1623.56,NaN,0,NaN,EM


# 1. Clean column names

In [167]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

# # using dict and zip
# new_data_columns = data.columns.str.strip().str.lower().str.replace(' ', '_')
# data.rename(columns=dict(zip(data.columns, new_data_columns)), inplace=True)

# # using lambda function
# data.rename(columns=lambda x: x.strip().lower().replace(' ', '_'), inplace=True)

# 2. Type conversion

In [168]:
data_spend_dirty = data['spend'].str.contains(r'\$')
print(data.loc[data_spend_dirty, ['campaign_id', 'spend']].head(5))

data['spend'] = data['spend'].str.replace(r'[^\d.]', '', regex=True)
data['spend'] = pd.to_numeric(data['spend'], errors='coerce')

print('FIXED!!!')
print(data.loc[data_spend_dirty, ['campaign_id', 'spend']].head(5))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
31   CMP-00032  $2759.35
32   CMP-00033  $2393.02
FIXED!!!
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22
31   CMP-00032  2759.35
32   CMP-00033  2393.02


# 3. Categorical columns

In [169]:
print(data['channel'].unique())

cleanup_map = {
    'TikTok': 'TikTok',
    'Facebook': 'Facebook',
    'Email': 'Email',
    'Instagram': 'Instagram',
    'Google Ads': 'Google Ads',
    'E-mail': 'Email',
    'nan': np.nan,
    'Gogle': 'Google Ads',
    'Tik_Tok': 'TikTok',
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram'
}

data['channel'] = data['channel'].replace(cleanup_map)

print('FIXED!!!')
print(data['channel'].unique())

<StringArray>
[    'TikTok',   'Facebook',      'Email',  'Instagram', 'Google Ads',
     'E-mail',          nan,      'Gogle',    'Tik_Tok',    'Facebok',
 'Insta_gram']
Length: 11, dtype: str
FIXED!!!
<StringArray>
['TikTok', 'Facebook', 'Email', 'Instagram', 'Google Ads', nan]
Length: 6, dtype: str


# 4. BOOL columns

In [170]:
print(data['active'].unique())

bool_cleanup = {
    'Y': True,
    '0': False,
    'No': False,
    'True': True,
    'Yes': True,
    '1': True,
    'False': False
}

data['active'] = data['active'].replace(bool_cleanup)

print('FIXED!!!')
print(data['active'].unique())

<StringArray>
['Y', '0', 'No', 'True', 'Yes', '1', 'False']
Length: 7, dtype: str
FIXED!!!
[True False]


# 5. Datetime columns

In [171]:
original_start_date = data['start_date']
original_end_date = data['end_date']

In [172]:
data['start_date'] = pd.to_datetime(data['start_date'], format='mixed', errors='coerce')
data['end_date'] = pd.to_datetime(data['end_date'], format='mixed', dayfirst=True, errors='coerce')

# 6. Logical integrity

## Click and impressions

In [173]:
# remove the duplicated columns and keep first
data = data.loc[:, ~data.columns.duplicated()]

In [174]:
# check if clicks are greater than impressions
check_mask = data['clicks'] > data['impressions']
print(data.loc[check_mask, ['campaign_id', 'clicks', 'impressions']])

Empty DataFrame
Columns: [campaign_id, clicks, impressions]
Index: []


## Time travel

In [175]:
check_data_mask = data['start_date'] > data['end_date']
data.loc[check_data_mask, ['campaign_id', 'start_date', 'end_date']]

,campaign_id,start_date,end_date
3,CMP-00004,2023-10-30,2023-03-11
8,CMP-00009,2023-03-23,2023-01-04
9,CMP-00010,2023-03-21,2023-01-04
10,CMP-00011,2023-02-22,2023-01-03
12,CMP-00013,2023-05-16,2023-01-06
...,...,...,...
1964,CMP-01965,2023-05-20,2023-05-06
1968,CMP-01969,2023-05-26,2023-05-06
1974,CMP-01975,2023-10-31,2023-10-11
1983,CMP-01984,2023-04-04,2023-02-05


In [176]:
# set end date to 30 days after start date if start date is after end date
data.loc[check_data_mask, 'end_date'] = data.loc[check_data_mask, 'start_date'] + pd.DateOffset(days=30)

# verify the changes
check_data_mask = data['start_date'] > data['end_date']
data.loc[check_data_mask, ['campaign_id', 'start_date', 'end_date']]

,campaign_id,start_date,end_date


## Handling outliners

In [177]:
q1 = data['spend'].quantile(0.25)
q3 = data['spend'].quantile(0.75)
IQR = q3 - q1

lower_bound = q1 - (3 * IQR)
upper_bound = q3 + (3 * IQR)

print(f'upper bound: {upper_bound:.2f} , lower bound: {lower_bound:.2f}')
outlier_mask = (data['spend'] < lower_bound) | (data['spend'] > upper_bound)
print(data.loc[outlier_mask, ['campaign_id', 'spend']])

upper bound: 8576.87 , lower bound: -5279.60
     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
1718   CMP-01719  500000.00
1754   CMP-01755  500000.00
1781   CMP-01782  500000.00


In [178]:
data.loc[outlier_mask, 'spend'] = upper_bound

## String extraction

In [181]:
data['campaign_name']

0            Q4_Summer_CMP-00001
1            Q1_Launch_CMP-00002
2            Q3_Winter_CMP-00003
3       Q1_BlackFriday_CMP-00004
4            Q2_Winter_CMP-00005
                  ...           
2015         Q3_Summer_CMP-00400
2016         Q4_Summer_CMP-01255
2017         Q2_Launch_CMP-01050
2018         Q4_Winter_CMP-01118
2019         Q4_Launch_CMP-01554
Name: campaign_name, Length: 2020, dtype: str

In [185]:
extract_season = data['campaign_name'].str.extract(r'_([^_]+)_')
extract_season

,0
0,Summer
1,Launch
2,Winter
3,BlackFriday
4,Winter
...,...
2015,Summer
2016,Summer
2017,Launch
2018,Winter


In [186]:
data['season'] = extract_season

In [188]:
data

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-12-05,Facebook,1860,30,24.33,1.0,False,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-29,TikTok,55886,2019,2180.38,135.0,True,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,False,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,False,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter


In [189]:
data.to_csv('cleaned_marketing_dataset.csv', index=False)